# 05 — Anomaly Resolution (M2)

Reads flagged Silver from Snowflake, applies **all Category A fixes** and **all Category B implementations**, and **overwrites corrected Silver back** to Snowflake.

**Idempotency contract:** running this notebook twice produces the same result. Corrections are guarded on the original `anomaly_reason_code` + `resolution_applied` so re-runs do not double-apply.

**Audit-trail contract (brief §8.1/§8.2):** every corrected row keeps `anomaly_flag=TRUE` and its original `anomaly_reason_code`, and gains `resolution_applied=TRUE` + `resolution_method` (+ `b_classification` for Category B). Nothing is deleted.

Order: Cat-A (A1–A16) first, then Cat-B (B1–B8), then write-back, then idempotency asserts.

In [ ]:
# --- Cell 1: install connector (Free Edition serverless; no Maven JAR) ---
%pip install -q snowflake-connector-python pandas rapidfuzz
dbutils.library.restartPython()

In [ ]:
# --- Cell 2: widgets + shared utils ---
import sys
sys.path.append('/Workspace/Repos/<your-org>/nexamart_m2/notebooks/_shared')  # TODO adjust path
dbutils.widgets.text('sf_account', '')
dbutils.widgets.text('sf_user', 'NEXAMART_LEAD')
dbutils.widgets.text('sf_password', '')
dbutils.widgets.text('sf_warehouse', 'NEXAMART_WH')
dbutils.widgets.text('sf_role', 'NEXAMART_ENGINEER')

from pyspark.sql import functions as F
import utils_snowflake as sf
import utils_anomaly as ua   # add_resolution_columns, resolve, assert_resolved_count, RESOLUTION_METHODS
import utils_dates as ud
import utils_keys as uk

## Resolution-column contract
For each Silver table you touch: `df = ua.add_resolution_columns(df)` once after reading, then apply the
data correction (e.g. zero a revenue column) **and** `df = ua.resolve(df, cond, '<METHOD>', b_classification=...)`.
Keep `anomaly_flag` / `anomaly_reason_code` untouched.

## Category A

In [ ]:
# === A1 — CANCELLED_WITH_REVENUE (owner M5) ===
# ec = sf.read_from_snowflake(spark, 'silver_ec_orders', 'NEXAMART_SILVER')
# ec = ua.add_resolution_columns(ec)
# cond = (F.col('order_status') == 'CANCELLED') & (F.col('subtotal_excl_tax') > 0)
# ec = ec.withColumn('subtotal_excl_tax', F.when(cond, F.lit(0)).otherwise(F.col('subtotal_excl_tax')))
# ec = ua.resolve(ec, cond, 'ZEROED_CANCELLED_REVENUE')
# TODO implement

In [ ]:
# === A2 — PAYMENT_AFTER_CANCEL (M5): flag reversal-required; method FLAGGED_REVERSAL_REQUIRED ===
# TODO implement

In [ ]:
# === A3 — TAX_INCLUSION_MISMATCH (M5): normalise tax-exclusive; method NORMALISED_TAX_EXCLUSIVE ===
# TODO implement

In [ ]:
# === A4 — NL_SELLER_SOLD_AS_REVENUE (M6): relabel ESTIMATED; method RELABELLED_ESTIMATED_GMV ===
# TODO implement

In [ ]:
# === A5 — ATP_POSITIVE_PHYSICAL_ZERO (Lead): ATP->0; method CORRECTED_ATP_TO_ZERO ===
# TODO implement

In [ ]:
# === A6 — NEGATIVE_QTY (M4): qty->0 + oversell flag; method CORRECTED_ATP_TO_ZERO ===
# TODO implement

In [ ]:
# === A7 — RESTOCK_BEFORE_INSPECTION (Lead): zero pre-inspection restock; ZEROED_RESTOCK_PRE_INSPECTION ===
# TODO implement

In [ ]:
# === A8 — MISSING_SNAPSHOT_DAY (Lead): reconstruct stores 3/7/12 x 7 days (1-7 Aug) ===
# last-known snapshot + intervening transactions; data_quality_status='RECONSTRUCTED', certainty INFERRED;
# method RECONSTRUCTED_SNAPSHOT. (Decide: merge sibling silver_store_inventory_snapshots_reconstructed.)
# TODO implement

In [ ]:
# === A9 — SKU_PRODUCT_MISMATCH (M3): canonical product (catalogue wins); APPLIED_CANONICAL_PRODUCT ===
# TODO implement

In [ ]:
# === A10 — OPEN_BOX_AS_NEW (M4): correct condition to open-box; quantify premium; CORRECTED_CONDITION_OPEN_BOX ===
# TODO implement

In [ ]:
# === A11 — PLACEHOLDER_ID_COLLISION (M2): rekey guests to GUEST-{session}; REKEYED_GUEST_BUCKET ===
# TODO implement (use uk.surrogate_key / anonymous_key)

In [ ]:
# === A12 — RELISTED_AFTER_SOLD (Lead): link pair; reliability=LOW; exclude from GMV; LINKED_RELISTING_EXCLUDED ===
# TODO implement (also feeds B6 exclusion)

In [ ]:
# === A13 — IMAGE_HASH_REUSED ring (M6): multi-signal flag + risk tier; FLAGGED_FRAUD_RING ===
# TODO implement

In [ ]:
# === A14 — DELIVERY_BEFORE_SHIP (Lead) ===
# delta<=72h: corrected_ts = PICKED_UP + 36h median (CORRECTED_DELIVERY_TS)
# delta>72h : ESCALATED_MANUAL_REVIEW (do NOT auto-correct)
# TODO implement

In [ ]:
# === A15 — REVIEW_BEFORE_DELIVERY (M6): verified_purchase=FALSE; SET_VERIFIED_PURCHASE_FALSE ===
# TODO implement

In [ ]:
# === A16 — DUPLICATE_CASE (M6): dedupe -> canonical_case_key; DEDUPED_CANONICAL_CASE ===
# TODO implement

## Category B — implement chosen interpretation + b_classification (defend in report S1)

In [ ]:
# === B1 — ATTRIBUTION (Lead): attribute 102 bridged orders; b_classification='ATTRIBUTED', conf 0.85 ===
# TODO implement

In [ ]:
# === B2 — PARTIAL_REFUND_PERIOD (Lead): recognise in return period; b_classification='RECOGNISE_IN_RETURN_PERIOD' ===
# TODO implement

In [ ]:
# === B3 — MOVEMENT_NULL_REF (Lead): b_classification='PROBABLE_MISSING_REF' (INFERRED) ===
# TODO implement

In [ ]:
# === B4 — LISTING match (M3): >=0.75 MATCHED / 0.65-0.75 MANUAL_REVIEW / <0.65 UNMATCHED (rapidfuzz) ===
# TODO implement

In [ ]:
# === B5 — IDENTITY merge (M2): probabilistic >=0.90 -> b_classification='MERGED' ===
# TODO implement

In [ ]:
# === B6 — ESTIMATED_NL_GMV (M2) ===
# weights SELLER_SOLD*0.60 + PHN_REVEAL*0.15 + CHAT*0.08 + OFFER_ACC*0.30, x listing confidence, +/-35% band.
# Exclude A12 relisted originals. Output lower/point/upper; ESTIMATED only.
# TODO implement

In [ ]:
# === B7 — BOPIS (Lead): b_classification='TREAT_AS_FULFILLED' + collection_unconfirmed flag ===
# TODO implement

In [ ]:
# === B8 — SELLER trust (M2): weighted composite -> 5 risk tiers; b_classification=tier; document weights ===
# TODO implement

## Write corrected Silver back (overwrite — idempotent)

In [ ]:
# One write per corrected table. Example:
# sf.write_to_snowflake(ec, 'silver_ec_orders', 'NEXAMART_SILVER', overwrite=True)
# Tables touched: silver_ec_orders, silver_pos_transactions, silver_pg_transactions,
#   silver_wh_inventory_snapshots, silver_si_inventory_snapshots, silver_rr_return_receipts,
#   silver_product_master, silver_customer_master, silver_nl_listings, silver_nl_listing_events,
#   silver_dc_delivery_events, silver_rv_reviews, silver_cs_cases, silver_ts_sellers,
#   silver_wh_inventory_movements, silver_rr_refund_events
# TODO write each

## Idempotency / acceptance asserts (re-run safe)

In [ ]:
# Re-read each corrected table and assert resolved counts match the targets in
# .private/resolution_targets.md. Example:
# ec2 = sf.read_from_snowflake(spark, 'silver_ec_orders', 'NEXAMART_SILVER')
# ua.assert_resolved_count(ec2, 'CANCELLED_WITH_REVENUE', 94)
# ua.assert_resolved_count(ec2, 'PLACEHOLDER_ID_COLLISION', 178)
# TODO assert all